In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import os
print(os.getcwd())
import sys, numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__, np.__file__)

In [ ]:
import glob
import os
import time
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from interface_analyzer import analyze_cfm, plot_cfm_k2_single
from interface_analyzer import PTMModifier, analyze_by_custom_modifier, CSPModifier

In [ ]:
import numpy as np
import pickle
import os
from pathlib import Path

# --- 1. Path setup ---
SAVE_DIR = FULL_DATA_ROOT / "110_001"
# Use a consistent variable name, Path_pkl
Path_pkl = SAVE_DIR / "110_001_cfg_post_orientation_grid_2_5_d_6_0.pkl"

# --- 2. Physical parameters ---
TEMPERATURE_K = 932.5
LATTICE_CONST_A = 4.134

# --- 3. Define the range and step size for the convergence analysis ---
STEP_SIZE = 500
MAX_FRAMES = 10000 
frame_counts = list(range(STEP_SIZE, MAX_FRAMES + 1, STEP_SIZE))

# Initialize result containers
convergence_results = []

# --- 4. Load the full raw data ---
print(f"Loading full dataset from {Path_pkl}...")
with open(Path_pkl, "rb") as f:
    results_full = pickle.load(f)

# Collect and sort all keys before entering the loop
all_available_keys = sorted(results_full.keys())
total_available = len(all_available_keys)
print(f"Total frames found in file: {total_available}")

# --- 5. Start the analysis loop ---
for count in frame_counts:
    # Check whether the requested number of frames exceeds the available frames
    if count > total_available:
        print(f"Skipping {count} frames: only {total_available} available.")
        continue
        
    print(f"\n>>> Analyzing first {count} frames...")

    # --- Robust slicing logic ---
    target_keys = all_available_keys[:count]
    results_subset = {k: results_full[k] for k in target_keys}
    
    print(f"Subset size: {len(results_subset)} frames (from step {target_keys[0]} to {target_keys[-1]})")

    # Save the temporary subset file
    temp_pkl_path = SAVE_DIR / f"temp_subset_{count}.pkl"
    with open(temp_pkl_path, "wb") as f:
        pickle.dump(results_subset, f)

    try:
        # --- Run the CFM analysis ---
        results_ptm = analyze_cfm(
            pickle_path=str(temp_pkl_path),
            T=TEMPERATURE_K,
            a=LATTICE_CONST_A,
            use_pchip=True,
            pchipres=10000,
            show_plot=False  # Set to False to avoid opening many figure windows
        )
        
        # --- Save the k^2 data ---
        output_base = SAVE_DIR / f"110_001_LOP_grid_2_5_d_6_0_{count}_frames"
        data_file = f"{output_base}_k2.dat"
        np.savetxt(
            data_file, 
            np.c_[results_ptm["k2"], results_ptm["Ak_min"], results_ptm["Ak_max"]], 
            fmt="%.8e", 
            header="k^2 Ak_min Ak_max"
        )

        # --- Linear fitting ---
        res_fit = plot_cfm_k2_single(
            data_file,
            label=f"{count} Frames",
            k_range=[0.07, 0.165],
            k2_min=5.0e-3,
            L_min_interface=4,
            min_points=6,
            through_origin=True,
            xlim=[0, 0.05],
            ylim=[0,0.6e-19]
        )

        # Extract Gamma and R2
        gamma = res_fit.get('gamma', 0) or res_fit.get('slope', 0)
        r2 = res_fit.get('r2', 0)
        convergence_results.append([count, gamma, r2])
        
        print(f"Successfully processed {count} frames. Gamma: {gamma:.6e}")

    except Exception as e:
        print(f"Error at {count} frames: {e}")
    finally:
        # Clean up temporary files
        if temp_pkl_path.exists():
            os.remove(temp_pkl_path)

# --- 6. Print final results ---
print("\n" + "="*55)
print(f"{'Frames':<10} | {'Gamma (Stiffness)':<20} | {'R-squared':<10}")
print("-" * 55)
for row in convergence_results:
    print(f"{row[0]:<10d} | {row[1]:<20.6e} | {row[2]:<10.6f}")
import csv    
with open("110_001_convergence_d6.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow(["N", "Error", "Value"])
    
    # Data
    writer.writerows(convergence_results)

In [ ]:
import csv    
with open("110_001_convergence_d6.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow(["N", "Error", "Value"])
    
    # Data
    writer.writerows(convergence_results)

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity
k2_data = SAVE_DIR / "110_001_LOP_grid_2_5_d_6_0_10000_frames_k2.dat"
fit_results = analyze_cfm_fit_sensitivity(
    k2_data,
    k2_min=5e-3,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
csp_fit_output = "csp_cfm_fit_sensitivity.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        csp_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )

In [ ]:
import numpy as np
import pickle
import os
from pathlib import Path

# --- 1. Path setup ---
SAVE_DIR = FULL_DATA_ROOT / "110_001"
# Use a consistent variable name, Path_pkl
Path_pkl = SAVE_DIR / "110_001_cfg_post_orientation_grid_2_5_d_8_0.pkl"

# --- 2. Physical parameters ---
TEMPERATURE_K = 932.5
LATTICE_CONST_A = 4.134

# --- 3. Define the range and step size for the convergence analysis ---
STEP_SIZE = 500
MAX_FRAMES = 10000 
frame_counts = list(range(STEP_SIZE, MAX_FRAMES + 1, STEP_SIZE))

# Initialize result containers
convergence_results = []

# --- 4. Load the full raw data ---
print(f"Loading full dataset from {Path_pkl}...")
with open(Path_pkl, "rb") as f:
    results_full = pickle.load(f)

# Collect and sort all keys before entering the loop
all_available_keys = sorted(results_full.keys())
total_available = len(all_available_keys)
print(f"Total frames found in file: {total_available}")

# --- 5. Start the analysis loop ---
for count in frame_counts:
    # Check whether the requested number of frames exceeds the available frames
    if count > total_available:
        print(f"Skipping {count} frames: only {total_available} available.")
        continue
        
    print(f"\n>>> Analyzing first {count} frames...")

    # --- Robust slicing logic ---
    target_keys = all_available_keys[:count]
    results_subset = {k: results_full[k] for k in target_keys}
    
    print(f"Subset size: {len(results_subset)} frames (from step {target_keys[0]} to {target_keys[-1]})")

    # Save the temporary subset file
    temp_pkl_path = SAVE_DIR / f"temp_subset_{count}.pkl"
    with open(temp_pkl_path, "wb") as f:
        pickle.dump(results_subset, f)

    try:
        # --- Run the CFM analysis ---
        results_ptm = analyze_cfm(
            pickle_path=str(temp_pkl_path),
            T=TEMPERATURE_K,
            a=LATTICE_CONST_A,
            use_pchip=True,
            pchipres=10000,
            show_plot=False  # Set to False to avoid opening many figure windows
        )
        
        # --- Save the k^2 data ---
        output_base = SAVE_DIR / f"110_001_LOP_grid_2_5_d_8_0_{count}_frames"
        data_file = f"{output_base}_k2.dat"
        np.savetxt(
            data_file, 
            np.c_[results_ptm["k2"], results_ptm["Ak_min"], results_ptm["Ak_max"]], 
            fmt="%.8e", 
            header="k^2 Ak_min Ak_max"
        )

        # --- Linear fitting ---
        res_fit = plot_cfm_k2_single(
            data_file,
            label=f"{count} Frames",
            k_range=[0.07, 0.165],
            k2_min=5.0e-3,
            L_min_interface=4,
            min_points=6,
            through_origin=True,
            xlim=[0, 0.05],
            ylim=[0,0.6e-19]
        )

        # Extract Gamma and R2
        gamma = res_fit.get('gamma', 0) or res_fit.get('slope', 0)
        r2 = res_fit.get('r2', 0)
        convergence_results.append([count, gamma, r2])
        
        print(f"Successfully processed {count} frames. Gamma: {gamma:.6e}")

    except Exception as e:
        print(f"Error at {count} frames: {e}")
    finally:
        # Clean up temporary files
        if temp_pkl_path.exists():
            os.remove(temp_pkl_path)

# --- 6. Print final results ---
print("\n" + "="*55)
print(f"{'Frames':<10} | {'Gamma (Stiffness)':<20} | {'R-squared':<10}")
print("-" * 55)
for row in convergence_results:
    print(f"{row[0]:<10d} | {row[1]:<20.6e} | {row[2]:<10.6f}")
import csv    
with open("110_001_convergence_d8.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow(["N", "Error", "Value"])
    
    # Data
    writer.writerows(convergence_results)

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity
k2_data = SAVE_DIR / "110_001_LOP_grid_2_5_d_8_0_10000_frames_k2.dat"
fit_results = analyze_cfm_fit_sensitivity(
    k2_data,
    k2_min=5e-3,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
csp_fit_output = "csp_cfm_fit_sensitivity_d8.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        csp_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )

In [ ]:
import numpy as np
import pickle
import os
from pathlib import Path

# --- 1. Path setup ---
SAVE_DIR = FULL_DATA_ROOT / "110_001"
# Use a consistent variable name, Path_pkl
Path_pkl = SAVE_DIR / "110_001_cfg_post_orientation_grid_2_5_d_4_0.pkl"

# --- 2. Physical parameters ---
TEMPERATURE_K = 932.5
LATTICE_CONST_A = 4.134

# --- 3. Define the range and step size for the convergence analysis ---
STEP_SIZE = 500
MAX_FRAMES = 10000 
frame_counts = list(range(STEP_SIZE, MAX_FRAMES + 1, STEP_SIZE))

# Initialize result containers
convergence_results = []

# --- 4. Load the full raw data ---
print(f"Loading full dataset from {Path_pkl}...")
with open(Path_pkl, "rb") as f:
    results_full = pickle.load(f)

# Collect and sort all keys before entering the loop
all_available_keys = sorted(results_full.keys())
total_available = len(all_available_keys)
print(f"Total frames found in file: {total_available}")

# --- 5. Start the analysis loop ---
for count in frame_counts:
    # Check whether the requested number of frames exceeds the available frames
    if count > total_available:
        print(f"Skipping {count} frames: only {total_available} available.")
        continue
        
    print(f"\n>>> Analyzing first {count} frames...")

    # --- Robust slicing logic ---
    target_keys = all_available_keys[:count]
    results_subset = {k: results_full[k] for k in target_keys}
    
    print(f"Subset size: {len(results_subset)} frames (from step {target_keys[0]} to {target_keys[-1]})")

    # Save the temporary subset file
    temp_pkl_path = SAVE_DIR / f"temp_subset_{count}.pkl"
    with open(temp_pkl_path, "wb") as f:
        pickle.dump(results_subset, f)

    try:
        # --- Run the CFM analysis ---
        results_ptm = analyze_cfm(
            pickle_path=str(temp_pkl_path),
            T=TEMPERATURE_K,
            a=LATTICE_CONST_A,
            use_pchip=True,
            pchipres=10000,
            show_plot=False  # Set to False to avoid opening many figure windows
        )
        
        # --- Save the k^2 data ---
        output_base = SAVE_DIR / f"110_001_LOP_grid_2_5_d_4_0_{count}_frames"
        data_file = f"{output_base}_k2.dat"
        np.savetxt(
            data_file, 
            np.c_[results_ptm["k2"], results_ptm["Ak_min"], results_ptm["Ak_max"]], 
            fmt="%.8e", 
            header="k^2 Ak_min Ak_max"
        )

        # --- Linear fitting ---
        res_fit = plot_cfm_k2_single(
            data_file,
            label=f"{count} Frames",
            k_range=[0.07, 0.165],
            k2_min=5.0e-3,
            L_min_interface=4,
            min_points=6,
            through_origin=True,
            xlim=[0, 0.05],
            ylim=[0,0.6e-19]
        )

        # Extract Gamma and R2
        gamma = res_fit.get('gamma', 0) or res_fit.get('slope', 0)
        r2 = res_fit.get('r2', 0)
        convergence_results.append([count, gamma, r2])
        
        print(f"Successfully processed {count} frames. Gamma: {gamma:.6e}")

    except Exception as e:
        print(f"Error at {count} frames: {e}")
    finally:
        # Clean up temporary files
        if temp_pkl_path.exists():
            os.remove(temp_pkl_path)

# --- 6. Print final results ---
print("\n" + "="*55)
print(f"{'Frames':<10} | {'Gamma (Stiffness)':<20} | {'R-squared':<10}")
print("-" * 55)
for row in convergence_results:
    print(f"{row[0]:<10d} | {row[1]:<20.6e} | {row[2]:<10.6f}")
import csv    
with open("110_001_convergence_d4.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # Header
    writer.writerow(["N", "Error", "Value"])
    
    # Data
    writer.writerows(convergence_results)

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity
k2_data = SAVE_DIR / "110_001_LOP_grid_2_5_d_4_0_10000_frames_k2.dat"
fit_results = analyze_cfm_fit_sensitivity(
    k2_data,
    k2_min=5e-3,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
csp_fit_output = "csp_cfm_fit_sensitivity_d8.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        csp_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )